# Philadelphia Passes the 10-Minute Test

A walk from every park edge along the streets, for parks of different sizes, against the city's Heat Vulnerability Index. Run `fetch_week1.py day5 day7` first.

## 1 · Settings

In [ ]:
DAY      = 7
PROMPT   = "10 minute map"
SLUG     = "ten-minute-parks-philly"
NOTEBOOK = "day07_ten_minute_parks_philly.ipynb"
TITLE    = "Philadelphia Passes\nthe 10-Minute Test"
CREDITS  = "Parks and Heat Vulnerability Index: City of Philadelphia. Streets: © OpenStreetMap contributors. Published 97% figure: Trust for Public Land."

## 2 · Load

In [ ]:
import geopandas as gpd, matplotlib.pyplot as plt, networkx as nx, numpy as np, pandas as pd, shapely
from matplotlib.collections import LineCollection
from mapkit import poster, publish
from mapkit.paths import PROCESSED

parks = gpd.read_file(PROCESSED / "tenmin_parks.geojson").to_crs(32618)
heat = gpd.read_file(PROCESSED / "heat_tracts.geojson").to_crs(32618)
streets = gpd.read_file(PROCESSED / "philly_streets.geojson").to_crs(32618)
water = gpd.read_file(PROCESSED / "philly_water.geojson").to_crs(3857)
city = gpd.read_file(PROCESSED / "places_six.geojson").query("city == 'philly'").to_crs(32618)

## 3 · Analysis, every number on the map comes from here

In [ ]:
TEN_MINUTES_M = 804.7       # half a mile, the distance the Trust for Public Land uses for a 10-minute walk
SIZES = {"any": 0, "1 acre": 1, "10 acres": 10, "50 acres": 50}

walkable = streets[streets.highway != "motorway"]
lines = shapely.get_parts(shapely.node(shapely.multilinestrings(shapely.get_parts(walkable.geometry.values))))
rows = []
G = nx.Graph()
for line in lines:
    xy = np.round(shapely.get_coordinates(line), 1)
    a, b = tuple(xy[0]), tuple(xy[-1])
    G.add_edge(a, b, length=shapely.length(line), geom=line)
nodes = gpd.GeoDataFrame({"node": list(G.nodes)}, crs=32618, geometry=gpd.points_from_xy(*zip(*G.nodes)))
print(f"Street network: {G.number_of_nodes():,} junctions, {G.number_of_edges():,} segments")

# Walk from every park edge outward along the streets, once for each park size
for name, acres in SIZES.items():
    door = gpd.GeoDataFrame(geometry=parks[parks.acres >= acres].buffer(25), crs=32618)
    sources = set(gpd.sjoin(nodes, door, predicate="within").node)
    dist = nx.multi_source_dijkstra_path_length(G, sources, weight="length")
    for u, v, d in G.edges(data=True):
        ends = [dist[n] for n in (u, v) if n in dist]
        d[name] = min(ends) + d["length"] / 2 if ends else np.inf

seg = gpd.GeoDataFrame([{"length": d["length"], **{k: d[k] for k in SIZES}, "geometry": d["geom"]} for _, _, d in G.edges(data=True)], crs=32618)
middle = gpd.GeoDataFrame(geometry=seg.geometry.interpolate(0.5, normalized=True), crs=32618)
inside = gpd.sjoin(middle, city[["geometry"]], predicate="within", how="inner").index
seg = seg.loc[inside]
middle = middle.loc[inside]
hot = heat[heat.n_veryhigh == 1]
seg["hot"] = middle.within(hot.union_all())

def share(frame, col):
    return float(frame[frame[col] <= TEN_MINUTES_M].length.sum() / frame.length.sum() * 100)

table = pd.DataFrame({name: {"all streets": share(seg, name), "most heat-vulnerable tracts": share(seg[seg.hot], name),
                             "other tracts": share(seg[~seg.hot], name)} for name in SIZES}).round(1)
print(table.to_string())
print(f"Hot tracts: {len(hot)} of {len(heat)}, {seg[seg.hot].length.sum() / seg.length.sum() * 100:.1f}% of street length")
print(f"Parks: {len(parks)}, {(parks.acres >= 1).sum()} of 1+ acre, {(parks.acres >= 10).sum()} of 10+, {(parks.acres >= 50).sum()} of 50+")
any_hot, big_hot, big_rest = table.loc["most heat-vulnerable tracts", "any"], table.loc["most heat-vulnerable tracts", "10 acres"], table.loc["other tracts", "10 acres"]

## 4 · Map

In [ ]:
PAPER, INK, GREEN, CLAY, PARK = "#f4f1e6", "#20302b", "#3f7d5a", "#c8552b", "#2f6b4c"
fig, ax = poster.page(PAPER)
m = seg.to_crs(3857)
near = m[m["10 acres"] <= TEN_MINUTES_M]
far = m[m["10 acres"] > TEN_MINUTES_M]
water[water.area > 3e5].plot(ax=ax, color="#d5e2e0", lw=0, zorder=1)
parks.to_crs(3857).plot(ax=ax, color=PARK, lw=0, alpha=0.32, zorder=2)
parks[parks.acres >= 10].to_crs(3857).plot(ax=ax, color=PARK, lw=0, zorder=3)
near.plot(ax=ax, color=GREEN, lw=0.32, zorder=4)
far.plot(ax=ax, color=CLAY, lw=0.85, zorder=4)
hot_m = hot.to_crs(3857)
hot_m.dissolve().boundary.plot(ax=ax, color=INK, lw=1.7, zorder=6)
ax.set_axis_off()
x0, y0, x1, y1 = city.to_crs(3857).total_bounds
fx0, fy0, fx1, fy1 = poster.fit_frame(ax, *city.to_crs(4326).total_bounds)
W, H = fx1 - fx0, fy1 - fy0
ax.set_xlim(fx0 - W * 0.04, fx1 + W * 0.04); ax.set_ylim(fy0 - H * 0.06, fy1 + H * 0.02)

fig.text(0.06, 0.965, "Philadelphia Passes", color=INK, fontproperties=poster.font(poster.DM_SERIF, 42), va="top")
fig.text(0.06, 0.915, "the 10-Minute Test", color=INK, fontproperties=poster.font(poster.DM_SERIF, 42), va="top")
fig.text(0.06, 0.862, f"In its most heat-vulnerable tracts, {any_hot:.0f}% of streets are a\n10-minute walk from a park. Only {big_hot:.0f}% reach one of 10 acres.",
         color=INK, fontproperties=poster.font(poster.BARLOW, 15), va="top", linespacing=1.35)

# the key, drawn with the same line weights as the map
key = fig.add_axes([0.6, 0.075, 0.36, 0.17]); key.set_axis_off(); key.set_xlim(0, 1); key.set_ylim(0, 4)
key.plot([0, .12], [3.5, 3.5], color=CLAY, lw=1.6); key.text(.16, 3.5, "Over 10 minutes from a 10-acre park", color=INK, fontproperties=poster.font(poster.BARLOW, 12), va="center")
key.plot([0, .12], [2.6, 2.6], color=GREEN, lw=0.6); key.text(.16, 2.6, "Within 10 minutes of one", color=INK, fontproperties=poster.font(poster.BARLOW, 12), va="center")
key.add_patch(plt.Rectangle((0, 1.5), .12, .4, color=PARK)); key.text(.16, 1.7, "Parks of 10 acres or more", color=INK, fontproperties=poster.font(poster.BARLOW, 12), va="center")
key.plot([0, .12], [.8, .8], color=INK, lw=1.7); key.text(.16, .8, "Most heat-vulnerable tracts", color=INK, fontproperties=poster.font(poster.BARLOW, 12), va="center")
poster.credit(fig, CREDITS + "\nShares measured by street length, along streets only.", path=poster.BARLOW, name_path=poster.BARLOW_BOLD, color="#55625c", name_color=INK, size=8.8)

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"{any_hot:.0f}% of streets in the most heat-vulnerable tracts reach a park in 10 minutes, {big_hot:.0f}% reach one of 10 acres",
             series="city", tool="Python", notebook=NOTEBOOK, credits=CREDITS)